### 🩺 Project Title: LLM-based Healthcare Assistant

## Introduction
This project demonstrates the use of Large Language Models (LLMs) in healthcare support. The model interacts with users, understands symptom descriptions, and predicts the most likely condition. It uses Retrieval-Augmented Generation (RAG) to reference medical documents, ensuring accurate and grounded responses. The goal is to build an AI-driven tool that offers safe, reliable, and efficient health guidance.

## Business Problem
Healthcare providers face challenges in handling high volumes of patient queries and ensuring consistent, timely, and accurate responses. Manual triage and advice are time-consuming and can lead to variability in quality. This project aims to automate symptom understanding and condition prediction to assist both patients and healthcare professionals.

## Challenges
Key challenges include processing large numbers of patient queries efficiently, maintaining accuracy and safety in condition predictions, handling ambiguous or incomplete symptom inputs, and integrating reliable medical references. Addressing these challenges motivates the design of a robust LLM-based healthcare assistant.- Handling large numbers of patient queries efficiently.


## Importing Libraries

In [5]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import re
from sklearn.preprocessing import LabelEncoder
# Install required libraries
!pip install -q transformers datasets torch accelerate sentencepiece
!pip install -q matplotlib seaborn pandas numpy
!pip install sentence-transformers faiss-cpu transformers
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    set_seed
)
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
set_seed(42)
np.random.seed(42)

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Step 1 - Load the Data 
The dataset containing symptom descriptions and corresponding disease labels is loaded using pandas. The unnecessary index column (Unnamed: 0) is removed to ensure data cleanliness. Displaying the first few rows provides an initial inspection of the text-label pairs, confirming that the data is correctly structured for subsequent preprocessing and model training.

In [7]:
df = pd.read_csv("Symptom2Disease.csv")
df = df.drop(['Unnamed: 0'], axis=1)
df.head()

,label,text
0,Psoriasis,I have been experiencing a skin rash on my arm...
1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,Psoriasis,I have been experiencing joint pain in my fing...
3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,Psoriasis,"My nails have small dents or pits in them, and..."


## Step 2 - Exploratory Data Analysis (EDA)
It is performed to assess dataset quality and characteristics. Missing values and duplicate entries are identified and removed to ensure data integrity. A new feature, text_length, is created to analyze the distribution of symptom description lengths, providing insight into text variability for model preparation.

In [9]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Missing values:
 label    0
text     0
dtype: int64

Duplicate rows: 47


In [10]:
df = df.drop_duplicates()
df['text_length'] = df['text'].apply(len)
df['text_length'].describe()

count    1153.000000
mean      171.112749
std        35.500946
min        60.000000
25%       147.000000
50%       169.000000
75%       192.000000
max       317.000000
Name: text_length, dtype: float64

# Step 3 - Preprocessing & Data Split
Text data is cleaned by removing leading and trailing whitespace to standardize inputs. Disease labels are encoded numerically using LabelEncoder to make them compatible with machine learning models, and a mapping is retained for interpretability. The dataset is stratified and split into training (80%), validation (10%), and test (10%) sets to ensure balanced label distribution and enable reliable model evaluation.

In [12]:

df['text'] = df['text'].str.strip()
encoder = LabelEncoder()
df['encoded_label'] = encoder.fit_transform(df['label'])

# Map numeric labels back to disease names if needed
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Label Mapping:", label_mapping)


Label Mapping: {'Acne': 0, 'Arthritis': 1, 'Bronchial Asthma': 2, 'Cervical spondylosis': 3, 'Chicken pox': 4, 'Common Cold': 5, 'Dengue': 6, 'Dimorphic Hemorrhoids': 7, 'Fungal infection': 8, 'Hypertension': 9, 'Impetigo': 10, 'Jaundice': 11, 'Malaria': 12, 'Migraine': 13, 'Pneumonia': 14, 'Psoriasis': 15, 'Typhoid': 16, 'Varicose Veins': 17, 'allergy': 18, 'diabetes': 19, 'drug reaction': 20, 'gastroesophageal reflux disease': 21, 'peptic ulcer disease': 22, 'urinary tract infection': 23}


In [13]:
# First split: train 80%, temp 20% (validation + test)
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['encoded_label'], 
    random_state=42
)

# Second split: validation 50%, test 50% of temp (so 10% each)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['encoded_label'],
    random_state=42
)

print("Train set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)


Train set: (922, 4)
Validation set: (115, 4)
Test set: (116, 4)


# Step 4: Baseline Model — TF-IDF + Logistic Regression

In [15]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))  # using unigrams + bigrams

# Fit on training text and transform train/validation/test
X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
X_test = tfidf.transform(test_df['text'])

y_train = train_df['encoded_label']
y_val = val_df['encoded_label']
y_test = test_df['encoded_label']

lr_model = LogisticRegression(max_iter=300, solver='lbfgs')
lr_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,300
,multi_class,'deprecated'


In [16]:
# Predictions on test set
y_pred = lr_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))


Test Accuracy: 0.9310

Classification Report:
                                 precision    recall  f1-score   support

                           Acne       1.00      1.00      1.00         4
                      Arthritis       1.00      1.00      1.00         5
               Bronchial Asthma       1.00      1.00      1.00         5
           Cervical spondylosis       1.00      1.00      1.00         5
                    Chicken pox       0.80      0.80      0.80         5
                    Common Cold       0.83      1.00      0.91         5
                         Dengue       0.75      0.60      0.67         5
          Dimorphic Hemorrhoids       1.00      1.00      1.00         4
               Fungal infection       1.00      1.00      1.00         5
                   Hypertension       1.00      1.00      1.00         5
                       Impetigo       1.00      1.00      1.00         5
                       Jaundice       1.00      1.00      1.00         4
    

A baseline classifier using TF-IDF vectorization and Logistic Regression was implemented to categorize symptom descriptions into medical conditions. The model achieved 93% accuracy, demonstrating strong performance on structured feature representations. However, it lacks the contextual and semantic understanding offered by LLM-based approaches, which are better suited for handling nuanced natural language inputs.

## Step 5: Qwen Healthcare Application

The pretrained Qwen3-0.6B LLM is loaded to classify symptom descriptions and generate safe health recommendations. The tokenizer converts natural language inputs into numerical tokens, while the model handles context-aware text generation. The implementation leverages GPU acceleration if available, ensuring efficient inference for free-text symptom inputs. This setup supports advanced NLP techniques including few-shot prompting, Chain-of-Thought reasoning, and Retrieval-Augmented Generation (RAG) for contextually grounded predictions


In [19]:
model_name = "Qwen/Qwen3-0.6B" 

# Load tokenizer and model
qwen_tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

`torch_dtype` is deprecated! Use `dtype` instead!


### 5.1 Helper Function for Text Generation

This function generates responses from Qwen using structured prompts and optional Chain-of-Thought reasoning for improved context understanding. It handles tokenization, decoding, and output cleaning to produce concise, safe health recommendations.


In [21]:
# Usual format for Qwen-based models
def generate_with_qwen(prompt, max_length=512, temperature=0.7, do_sample=True, enable_thinking=False):
    messages = [{"role": "user", "content": prompt}]
    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    model_inputs = qwen_tokenizer([text], return_tensors="pt").to("cpu")  # force CPU

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **model_inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = qwen_tokenizer.decode(output_ids, skip_special_tokens=True)
    return response.split("</think>")[-1].lstrip() if "</think>" in response else response.strip()

# Test prompt
prompt = "I have fever, sore throat, and cough"
response = generate_with_qwen(prompt)
print(prompt)
print("Generated Response:\n", response)


I have fever, sore throat, and cough
Generated Response:
 It's important to take care of yourself while feeling unwell. Here are some general recommendations:

1. **Rest**: Make sure you're resting as much as possible.
2. **Hydration**: Drink plenty of water to keep your body hydrated.
3. **Humidifier or steam**: Use a humidifier or sit in steam to help with congestion.
4. **Medication**: If your symptoms persist, consult a healthcare provider for appropriate treatment. They may prescribe acetaminophen or ibuprofen if needed.
5. **Avoid triggers**: Avoid spicy or hot foods, cough syrup, and other irritants that can worsen symptoms.
6. **Monitor symptoms**: If your symptoms get worse, seek medical attention immediately.

Please remember that if you're experiencing severe symptoms (e.g., high fever, difficulty breathing, chest pain), you should go to the emergency room or a hospital.


### 5.2 Symptom Classification using Qwen [PROMPTING]
The Qwen model is prompted to classify patient symptoms into predefined conditions using a structured instruction. This approach ensures precise, single-label predictions while leveraging the LLM’s contextual understanding of natural language inputs.

In [23]:
# Define possible conditions
possible_conditions = [
    "Psoriasis", "Eczema", "Acne", "Allergy", "Diabetes",
    "Hypertension", "Common Cold", "Migraine", "Malaria",
    "Dengue", "Fungal infection", "Bronchial Asthma", "Typhoid",
    "Chicken pox", "Cervical spondylosis", "Pneumonia",
    "Urinary tract infection", "Drug reaction", "Peptic ulcer disease",
    "Dimorphic Hemorrhoids", "Jaundice", "Gastroesophageal reflux disease",
    "Arthritis", "Impetigo"
]

# --- Step 1: Classify symptoms ---
def classify_symptom_with_qwen(symptom_text, possible_conditions):
    prompt = f"""
The patient describes these symptoms: "{symptom_text}"
From the following list, pick the most probable condition: {possible_conditions}.
Return ONLY the condition name. Do NOT include reasoning, explanations, or <think> text.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.7, do_sample=False)
    return response.strip()

prompt = "I have fever, sore throat, and cough"
response = classify_symptom_with_qwen (prompt, possible_conditions)
print(prompt)
print("Generated Response:\n", response)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


I have fever, sore throat, and cough
Generated Response:
 Common Cold


### 5.3 Generate safe recommendation
Using the predicted condition from Qwen, the model generates a concise, safe, and neutral health recommendation for the patient. This ensures that LLM outputs are contextually relevant while maintaining safety and clarity for end users.

In [25]:
# --- Step 2: Generate safe recommendation ---
def generate_health_recommendation(symptom_text, condition):
    prompt = f"""
The patient has the following symptoms: "{symptom_text}"
The probable condition is: "{condition}"
Write a short, safe, and neutral health recommendation or advice for the patient.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.5, do_sample=True)
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    return response.strip()

prompt = "The patient has a red, itchy rash on their arms and legs."

# First, classify to get the condition
predicted_condition = classify_symptom_with_qwen(prompt, possible_conditions)

# Then, generate health recommendation using the symptom text and predicted condition
response = generate_health_recommendation(prompt, predicted_condition)

print("Prompt:\n", prompt)
print("\nPredicted Condition:\n", predicted_condition)
print("\nGenerated Recommendation:\n", response)

Prompt:
 The patient has a red, itchy rash on their arms and legs.

Predicted Condition:
 Eczema

Generated Recommendation:
 The patient likely has eczema. A safe and neutral recommendation is to advise the patient to follow a gentle skincare routine, avoid scratching the affected areas, and consult a healthcare provider if the rash persists or worsens.


In [26]:
# --- Step 3: Combine into one function for Gradio ---

def healthcare_chat(user_input):
    condition = classify_symptom_with_qwen(user_input, possible_conditions)
    recommendation = generate_health_recommendation(user_input,condition)
    return condition, recommendation

### 5.5 Real-Time AI Health Assistant (Basic)
A Gradio interface is implemented to enable users to enter symptoms, receive predicted conditions, and view safe health recommendations in real time.

In [28]:
import gradio as gr

iface = gr.Interface(
    fn=healthcare_chat,
    inputs=gr.Textbox(
        lines=3, 
        placeholder="Describe your symptoms here..."
    ),
    outputs=["text", "text"],
    title="🩺 AI Health Assistant",
    description="Enter your symptoms to get a possible condition and health recommendation using Qwen3-0.6B."
)

# --- Step 5: Launch app ---
if __name__ == "__main__":
    iface.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### STEP 6 - Test Cases 

This section tests the AI Health Assistant on multiple example symptoms to validate prediction accuracy and recommendation quality.
It demonstrates the system’s ability to handle diverse symptom inputs in a real-world scenario.

In [30]:
# Example test cases
test_cases = [
    "I have a red, itchy rash on my arms and legs.",
    "I am feeling cold and have a mild fever.",
    "I have a severe headache and nausea.",
    "My joints are painful and swollen.",
    "I have heartburn after meals frequently."
]

# Loop through test cases
for i, symptoms in enumerate(test_cases, 1):
    condition, recommendation = healthcare_chat(symptoms)  # unpack tuple
    print(f"--- Test Case {i} ---")
    print("Input Symptoms:", symptoms)
    print("Predicted Condition:", condition)
    print("Health Recommendation:", recommendation)
    print("\n")


--- Test Case 1 ---
Input Symptoms: I have a red, itchy rash on my arms and legs.
Predicted Condition: Eczema
Health Recommendation: The probable condition is **eczema**.  

For a safe and neutral recommendation, you might say:  

"Based on your description of a red, itchy rash on your arms and legs, it is likely eczema. Please consult a healthcare provider for a proper diagnosis and treatment plan."


--- Test Case 2 ---
Input Symptoms: I am feeling cold and have a mild fever.
Predicted Condition: Common Cold
Health Recommendation: The patient likely has the common cold, which is characterized by a mild fever and feeling cold. A safe and neutral recommendation is to rest and take a warm bath or drink a warm beverage to help alleviate symptoms.


--- Test Case 3 ---
Input Symptoms: I have a severe headache and nausea.
Predicted Condition: Migraine
Health Recommendation: The patient likely has a severe headache and nausea, which could indicate a migraine. It is important to seek medical

## LLM TECHNIQUES 
### The RAG (Retrieval-Augmented Generation)
This method retrieves relevant medical documents to provide contextually grounded predictions and safe health recommendations. By incorporating verified domain knowledge into the prompt, it reduces hallucinations, enhances the LLM’s understanding of symptom descriptions, and improves the overall accuracy and reliability of condition predictions and advice.

In [32]:
# Sample healthcare knowledge base (RAG-style)
health_docs = [
    'Acne is a skin condition that causes pimples or "zits." Whiteheads, blackheads, and red, inflamed patches of skin (such as cysts) may develop.',
    'Eczema (atopic dermatitis) is a common skin condition that causes itchy skin. It affects people of all ages but is most common in young children. It cannot be cured, but treatment can help manage the symptoms.',
    'Psoriasis is a skin disease that causes itchy or sore patches of thick, red skin with silvery scales. You usually get the patches on your elbows, knees, scalp, back, face, palms and feet, but they can show up on other parts of your body. Some people who have psoriasis also get a form of arthritis called psoriatic arthritis.',
    'Migraines are a recurring type of headache. They cause moderate to severe pain that is throbbing or pulsing. The pain is often on one side of your head. You may also have other symptoms, such as nausea and weakness. You may be sensitive to light and sound.',
    'Gastroesophageal reflux disease (GERD) happens when a muscle at the end of your esophagus does not close properly. This allows stomach contents to leak back, or reflux, into the esophagus and irritate it.',
    'Jaundice causes your skin and the whites of your eyes to turn yellow. Too much bilirubin causes jaundice. Bilirubin is a yellow chemical in hemoglobin...',
    'Impetigo is a skin infection caused by bacteria. It is most common in children between the ages of two and six.',
    'Arthritis causes pain and stiffness in joints.',
    'Diabetes, also known as diabetes mellitus, is a disease in which your blood glucose levels are too high.',
    'The common cold is a mild infection of your upper respiratory tract.',
    'Malaria is a serious disease caused by a parasite transmitted by mosquitoes.',
    'Urinary tract infections (UTIs) affect the bladder and urinary tract.',
    'Allergy is a reaction by the immune system to substances that don’t affect most people.',
    'Asthma is a chronic lung disease affecting the airways.',
    'Chickenpox is an infection caused by the varicella-zoster virus.'
]




In [33]:
# --- Step 0: RAG keyword retrieval ---
def retrieve_health_docs(symptom_text, health_docs, top_k=1):
    """
    Simple keyword matching retrieval.
    Returns the top_k most relevant documents based on keyword overlap.
    """
    symptom_words = set(symptom_text.lower().split())
    doc_scores = []

    for doc in health_docs:
        doc_words = set(doc.lower().split())
        score = len(symptom_words & doc_words)  # count of overlapping words
        doc_scores.append((score, doc))

    # Sort by score descending
    doc_scores.sort(reverse=True, key=lambda x: x[0])

    # Return top_k documents
    top_docs = [doc for score, doc in doc_scores[:top_k] if score > 0]
    return top_docs if top_docs else [""]  # return empty string if no match


# --- Step 1: Modify classify_symptom_with_qwen to include RAG docs ---
def classify_symptom_with_rag(symptom_text, possible_conditions, health_docs):
    retrieved_docs = retrieve_health_docs(symptom_text, health_docs, top_k=1)
    retrieved_text = retrieved_docs[0]

    prompt = f"""
The patient describes these symptoms: "{symptom_text}"
Reference info: "{retrieved_text}"
From the following list, pick the most probable condition: {possible_conditions}.
Return ONLY the condition name.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.7, do_sample=False)
    return response.strip()


# --- Step 2: Modify health recommendation to include RAG docs ---
def generate_health_recommendation_rag(symptom_text, condition, health_docs):
    retrieved_docs = retrieve_health_docs(symptom_text, health_docs, top_k=1)
    retrieved_text = retrieved_docs[0]

    prompt = f"""
The patient has the following symptoms: "{symptom_text}"
The probable condition is: "{condition}"
Reference info: "{retrieved_text}"
Write a short, safe, and neutral health recommendation or advice for the patient.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.5, do_sample=True)
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    return response.strip()


# --- Step 3: Combine into one function ---
def healthcare_chat_rag(user_input):
    condition = classify_symptom_with_rag(user_input, possible_conditions, health_docs)
    recommendation = generate_health_recommendation_rag(user_input, condition, health_docs)
    return condition, recommendation

test_input = "I have fever, sore throat, and cough"
condition, recommendation = healthcare_chat_rag(test_input)

print("🩺 Predicted Condition:", condition)
print("💡 Recommendation:", recommendation)

🩺 Predicted Condition: Common Cold
💡 Recommendation: The patient likely has a common cold, which is characterized by fever, sore throat, and cough. A safe and neutral recommendation is to rest, drink plenty of fluids, and consult a healthcare provider if symptoms persist or worsen.


### Few Shot Prompting 
- Few-shot prompting is used to guide the LLM using example symptom–condition pairs. The model classifies new symptoms based on these examples and generates safe, neutral health recommendations. This approach improves prediction accuracy and contextual relevance by providing the LLM with prior examples, allowing it to adapt to patterns in user inputs.

In [35]:
def classify_symptom_few_shot(symptom_text, possible_conditions):
    prompt = f"""
Here are some examples of symptoms and their most probable condition:
1. Symptoms: "I have a red, itchy rash on my arms and legs" → Condition: "Eczema"
2. Symptoms: "I have high fever, chills, and severe headache" → Condition: "Malaria"
3. Symptoms: "I have frequent urination and excessive thirst" → Condition: "Diabetes"
4. Symptoms: "I have runny nose, mild fever, and sore throat" → Condition: "Common Cold"
5. Symptoms: "I have headache and nausea after skipping meals" → Condition: "Migraine"

Now, classify the following symptoms from the list of possible conditions: {possible_conditions}
Symptoms: "{symptom_text}"
Return ONLY the most probable condition.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.7, do_sample=False)
    return response.strip()

def generate_health_recommendation_few_shot(symptom_text, condition):
    prompt = f"""
Examples:
1. Symptoms: "I have a red, itchy rash on my arms and legs" → Condition: "Eczema"
2. Symptoms: "I have high fever, chills, and severe headache" → Condition: "Malaria"
3. Symptoms: "I have frequent urination and excessive thirst" → Condition: "Diabetes"
4. Symptoms: "I have runny nose, mild fever, and sore throat" → Condition: "Common Cold"
5. Symptoms: "I have headache and nausea after skipping meals" → Condition: "Migraine"

Now generate a short, safe, neutral health recommendation for the following:
Symptoms: "{symptom_text}"
Condition: "{condition}"
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.5, do_sample=True)
    return response.strip()

user_symptoms = "I have fever, sore throat, and cough"

# Classify condition
predicted_condition = classify_symptom_few_shot(user_symptoms, possible_conditions)

# Generate recommendation
recommendation = generate_health_recommendation_few_shot(user_symptoms, predicted_condition)

print("Symptoms:", user_symptoms)
print("Predicted Condition:", predicted_condition)
print("Health Recommendation:", recommendation)


Symptoms: I have fever, sore throat, and cough
Predicted Condition: Common Cold
Health Recommendation: Symptoms: "I have fever, sore throat, and cough" → Condition: "Common Cold"


### logical reasonging with COT
- Chain-of-Thought prompting allows the LLM to reason step-by-step before providing a final prediction or recommendation. For symptom classification, the model first outlines its reasoning process and then outputs the most probable condition. Similarly, for health recommendations, it explains its reasoning before giving safe, neutral advice. This enhances interpretability, transparency, and contextual understanding, helping users and healthcare practitioners trust the AI’s suggestions.

In [37]:

def classify_symptom_with_cot(symptom_text, possible_conditions):
    prompt = f"""
The patient describes these symptoms: "{symptom_text}"
Think through the possible conditions step by step, then choose the most probable condition from the following list: {possible_conditions}.
Return your reasoning first, then the final condition in the format:
Reasoning: <your reasoning>
Condition: <final condition>
"""
    response = generate_with_qwen(prompt, max_length=200, temperature=0.7, do_sample=True)
    return response.strip()
    
def generate_health_recommendation_cot(symptom_text, condition):
    prompt = f"""
The patient has the following symptoms: "{symptom_text}"
The probable condition is: "{condition}"
Think step by step about what advice is safe and appropriate, then give a short, neutral recommendation.
Format:
Reasoning: <your reasoning>
Recommendation: <your recommendation>
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.5, do_sample=True)
    return response.strip()

def healthcare_chat_cot(user_input):
    cot_response = classify_symptom_with_cot(user_input, possible_conditions)
    # Extract the final condition from CoT output
    final_condition = cot_response.split("Condition:")[-1].strip()
    
    recommendation = generate_health_recommendation_cot(user_input, final_condition)
    return final_condition, recommendation


In [38]:
user_symptoms = "I have fever, sore throat, and cough"

# Get CoT classification
cot_output = classify_symptom_with_cot(user_symptoms, possible_conditions)

# Extract reasoning and final condition
reasoning_part = cot_output.split("Condition:")[0].replace("Reasoning:", "").strip()
final_condition = cot_output.split("Condition:")[-1].strip()

# Generate recommendation using the final condition
recommendation_output = generate_health_recommendation_cot(user_symptoms, final_condition)

# Extract recommendation reasoning and text
rec_reasoning = recommendation_output.split("Recommendation:")[0].replace("Reasoning:", "").strip()
rec_text = recommendation_output.split("Recommendation:")[-1].strip()

# Print nicely
print("User Symptoms:", user_symptoms)
print("\n--- Condition Prediction ---")
print("Reasoning:", reasoning_part)
print("Predicted Condition:", final_condition)
print("\n--- Health Recommendation ---")
print("Reasoning:", rec_reasoning)
print("Recommendation:", rec_text)


User Symptoms: I have fever, sore throat, and cough

--- Condition Prediction ---
Reasoning: The symptoms described (fever, sore throat, cough) are typical of a common cold. It is the most likely condition among the listed options.
Predicted Condition: Common Cold

--- Health Recommendation ---
Reasoning: The patient presents with fever, sore throat, and cough, which are typical symptoms of a common cold. These symptoms are not uncommon and typically resolve within a few days. The recommended action is to rest, drink fluids, and take over-the-counter medications as needed.
Recommendation: Rest and take rest.


### 5.4 Interactive Feedback Mechanism
The system incorporates user feedback to refine future predictions using few-shot prompting from past corrections. Each new feedback entry (correct or corrected condition) is stored in a feedback history, which is used as contextual examples for subsequent symptom classifications. By leveraging these few-shot examples, the LLM adapts over time, improving the accuracy of predictions while maintaining a transparent record of corrections. This mechanism simulates a human-in-the-loop learning process, allowing continuous model improvement in a controlled and interpretable manne

In [40]:
feedback_history = []

def classify_with_feedback(symptom_text):
    # Use last 5 feedbacks as few-shot examples
    examples = ""
    for f in feedback_history[-5:]:
        examples += f"Symptoms: {f['input']} Condition: {f['correct']}\n"
    
    prompt = f"""
You are a medical assistant. Here are some past corrections:
{examples}
Classify the following symptoms: "{symptom_text}" 
Return ONLY the most probable condition.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.7, do_sample=False)
    return response.strip()

def record_feedback(user_input, predicted_condition, feedback, correct_condition=None):
    # If user marks incorrect, they must provide correct condition
    final_condition = predicted_condition if feedback=="Correct ✅" else correct_condition
    feedback_history.append({
        "input": user_input,
        "predicted": predicted_condition,
        "correct": final_condition
    })
    return f"✅ Feedback recorded: '{feedback}' for condition '{predicted_condition}'"


# Update the Gradio submit function
def healthcare_chat_with_feedback(user_input):
    condition = classify_with_feedback(user_input)
    recommendation = generate_health_recommendation(user_input, condition)
    return condition, recommendation

# --- Simulate first classification ---
user_input1 = "I have fever, sore throat, and cough"
predicted_condition1 = classify_with_feedback(user_input1)
print("User Input:", user_input1)
print("Predicted Condition:", predicted_condition1)

# --- Record feedback ---
feedback1 = "Incorrect ❌"  # Simulate user saying the prediction was wrong
feedback_response1 = record_feedback(user_input1, predicted_condition1, feedback1)
print(feedback_response1)

# --- Simulate second classification using few-shot feedback ---
user_input2 = "My throat is sore and I have high fever"
predicted_condition2 = classify_with_feedback(user_input2)
print("\nUser Input:", user_input2)
print("Predicted Condition (with few-shot):", predicted_condition2)

# --- Record feedback again ---
feedback2 = "Correct ✅"
feedback_response2 = record_feedback(user_input2, predicted_condition2, feedback2)
print(feedback_response2)

# --- View feedback history ---
print("\nFeedback History:")
for f in feedback_history:
    print(f)


User Input: I have fever, sore throat, and cough
Predicted Condition: Fever, sore throat, and cough.
✅ Feedback recorded: 'Incorrect ❌' for condition 'Fever, sore throat, and cough.'

User Input: My throat is sore and I have high fever
Predicted Condition (with few-shot): High fever and sore throat are symptoms of a viral infection, such as the common cold or flu.
✅ Feedback recorded: 'Correct ✅' for condition 'High fever and sore throat are symptoms of a viral infection, such as the common cold or flu.'

Feedback History:
{'input': 'I have fever, sore throat, and cough', 'predicted': 'Fever, sore throat, and cough.', 'correct': None}
{'input': 'My throat is sore and I have high fever', 'predicted': 'High fever and sore throat are symptoms of a viral infection, such as the common cold or flu.', 'correct': 'High fever and sore throat are symptoms of a viral infection, such as the common cold or flu.'}


# Gradio Interface ( Feedback )
The AI Health Assistant interface allows users to enter symptoms, receive a predicted condition, and get a safe health recommendation. Users can provide feedback on predictions, which is stored for future adaptation. A dedicated tab displays the history of feedback, enabling easy tracking of corrections. The interface is organized using tabs for diagnosis, feedback submission, and feedback history, providing a clean and interactive experience.

In [42]:
import gradio as gr

# --- Step 1: Classify symptoms ---
def classify_symptom_with_qwen(symptom_text, possible_conditions):
    prompt = f"""
The patient describes these symptoms: "{symptom_text}"
From the following list, pick the most probable condition: {possible_conditions}.
Return ONLY the condition name. Do NOT include reasoning, explanations, or <think> text.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.7, do_sample=True)
    return response.strip()

# --- Step 2: Generate safe recommendation ---
def generate_health_recommendation(symptom_text, condition):
    prompt = f"""
The patient has the following symptoms: "{symptom_text}"
The probable condition is: "{condition}"
Write a short, safe, and neutral health recommendation or advice for the patient.
"""
    response = generate_with_qwen(prompt, max_length=150, temperature=0.5, do_sample=True)
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    return response.strip()


def healthcare_chat(user_input):
    condition = classify_symptom_with_qwen(user_input, possible_conditions)
    recommendation = generate_health_recommendation(user_input, condition)
    return condition, recommendation

feedback_history = []

def record_feedback(user_input, predicted_condition, feedback, correct_condition=None):
    final_condition = predicted_condition if feedback=="Correct ✅" else correct_condition
    feedback_history.append({
        "input": user_input,
        "predicted": predicted_condition,
        "correct": final_condition
    })
    return f"✅ Feedback recorded: '{feedback}' for condition '{predicted_condition}'"

def get_feedback_history():
    return "\n".join([f"{i+1}. Input: {f['input']}, Predicted: {f['predicted']}, Correct: {f['correct']}" 
                      for i, f in enumerate(feedback_history)]) or "No feedback yet."

# --- Gradio Interface ---
with gr.Blocks() as demo:
    gr.Markdown("## 🩺 AI Health Assistant")
    
    with gr.Tab("Symptom Diagnosis"):
        user_input = gr.Textbox(label="Describe your symptoms", placeholder="e.g., I have a rash on my arms")
        condition_output = gr.Textbox(label="Predicted Condition")
        recommendation_output = gr.Textbox(label="Health Recommendation")
        submit_btn = gr.Button("Get Diagnosis")
        
        def run_diagnosis(symptoms):
            condition,recommendation = healthcare_chat(symptoms)
            return condition, recommendation
        
        submit_btn.click(run_diagnosis, inputs=user_input, outputs=[condition_output, recommendation_output])
    
    with gr.Tab("Feedback"):
        feedback_input = gr.Radio(["Correct ✅", "Incorrect ❌"], label="Was the prediction correct?")
        correct_condition_input = gr.Textbox(label="Correct Condition (if incorrect)", placeholder="Type only if marked incorrect")
        feedback_msg = gr.Textbox(label="Feedback Message", interactive=False)
        feedback_btn = gr.Button("Submit Feedback")
        
        feedback_btn.click(
            record_feedback, 
            inputs=[user_input, condition_output, feedback_input, correct_condition_input],
            outputs=feedback_msg
        )
    
    with gr.Tab("Feedback History"):
        history_output = gr.Textbox(label="Past Feedbacks", interactive=False)
        refresh_btn = gr.Button("Refresh History")
        refresh_btn.click(get_feedback_history, inputs=[], outputs=history_output)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
